# Session Attendance Merger

Merges several session sheets into one Master workbook (one row per person, one column per session).

**How to use:** run the cells from top to bottom. Each file has its own upload cell. If you do not have a file, cancel its upload dialog and continue.

**What you get:** `Master_Attendance.xlsx` with the Master sheet plus review sheets (attendance conflicts, evidence for every True/False, ID conflicts, name review, possible duplicates).

## 1. Settings

In [ ]:
# ============================================================
# SECTION 1: Settings and upload helper
# ============================================================
# Edit the values in this cell if you need to. Everything else can run as-is.

from google.colab import files

# Attendance sessions in chronological order. Each name becomes a column in the output.
SESSIONS = ["Web Dev 1", "AI Session", "Python Workshop", "SWE 1", "Web Dev 2"]

# Reference file (registration only, no attendance). It is used to confirm names, IDs, emails
# and phones. It never creates attendance values.
REFERENCE_LABEL = "DS Session 1 (reference)"

# What to do when the same person has BOTH a True row and a False row in the same session:
#   "check"   -> write CHECK in the cell and list the case in the 'Attendance conflicts' sheet (recommended)
#   "present" -> treat as attended (any True wins)
#   "absent"  -> treat as not attended (any False wins)
MIXED_POLICY = "present"  # Changed from "check" to "present" per owner confirmation

# Manual decisions for specific people. Use ANY of the person's IDs (university or national).
# Example: OVERRIDES = {("251009956", "AI Session"): False}
# The 'Attendance conflicts' sheet has an 'Override key' column you can copy from.
OVERRIDES = {}

# Optional session dates (YYYY-MM-DD). If given, registrations submitted AFTER the date are flagged.
# Example: SESSION_DATES = {"AI Session": "2026-10-27"}
SESSION_DATES = {}

# If True, people who appear ONLY in the reference file are added to the Master sheet too.
# If False, they are listed in a separate 'Reference only' sheet.
INCLUDE_REFERENCE_ONLY_PEOPLE = False

# Name matching: how similar two names must be to be linked automatically (name-only rows).
NAME_LINK_THRESHOLD = 0.9

# Holds the uploaded file names: {label: file_name}
SESSION_FILES = {}

def upload_session(label):
    """Open an upload dialog for one file and remember its name under the given label."""
    print(f"Upload the file for: {label}")
    try:
        uploaded = files.upload()
    except Exception:
        uploaded = {}
    if uploaded:
        SESSION_FILES[label] = list(uploaded.keys())[0]
        print(f"OK: {label} <- {SESSION_FILES[label]}")
    else:
        print(f"Skipped: {label}")


## 2. Upload the files (one cell per file)

In [ ]:
# Upload: Web Dev 1
upload_session(SESSIONS[0])

In [ ]:
# Upload: AI Session
upload_session(SESSIONS[1])

In [ ]:
# Upload: Python Workshop
upload_session(SESSIONS[2])

In [ ]:
# Upload: SWE 1
upload_session(SESSIONS[3])

In [ ]:
# Upload: Web Dev 2
upload_session(SESSIONS[4])

In [ ]:
# Upload: reference registration file (no attendance data)
upload_session(REFERENCE_LABEL)

## 3. Core code (run without changes)

In [ ]:
# ============================================================
# SECTION 2: Cleaning helpers
# Each helper turns a messy raw value into a standard form so that
# the same person written in different ways can be matched.
# ============================================================
import re
import difflib
import pandas as pd

def is_blank(v):
    """True for None / NaN / empty string."""
    return v is None or (isinstance(v, float) and pd.isna(v)) or str(v).strip() == ""

# Arabic-Indic and Persian digits -> ASCII digits (some people type IDs with Arabic numerals)
_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")

def to_digits(v):
    """Keep digits only. Floats such as 221018080.0 become '221018080'. Arabic numerals are converted."""
    if is_blank(v):
        return ""
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return re.sub(r"[^0-9]", "", str(v).translate(_DIGITS))

def norm_email(v):
    """Lower-case and strip. Values without '@' are treated as empty."""
    if is_blank(v):
        return ""
    s = str(v).strip().lower()
    return s if "@" in s else ""

def norm_phone(v):
    """Standardise Egyptian mobiles to 01XXXXXXXXX. Other numbers keep their digits."""
    d = to_digits(v)
    if not d:
        return ""
    if d.startswith("0020"):
        d = d[4:]
    elif d.startswith("20") and len(d) == 12:
        d = d[2:]
    if len(d) == 10 and d.startswith("1"):      # leading zero was lost
        d = "0" + d
    return d if len(d) >= 9 else ""

def split_reg(v):
    """Registration field -> (university_id, national_id). 9 digits = university ID, 14 digits = national ID."""
    d = to_digits(v)
    if len(d) == 14:
        return "", d
    if len(d) == 9:
        return d, ""
    return "", ""

_AR = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ة": "ه", "ى": "ي", "ؤ": "و", "ئ": "ي"})

def norm_name(v):
    """Normalise a name for comparison only (lower-case, no punctuation/diacritics, unified Arabic letters)."""
    if is_blank(v):
        return ""
    s = str(v).strip().lower().translate(_AR)
    s = re.sub(r"[\u064B-\u065F\u0640]", "", s)
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def to_bool(v):
    """Attendance value -> True/False. Accepts 1/0, True/False, yes/no."""
    if isinstance(v, bool):
        return v
    if is_blank(v):
        return False
    return str(v).strip().lower() in ("1", "1.0", "true", "yes", "present")

def tokens(n):
    return set(n.split())

def name_similar(a, b):
    """0..1 similarity. A name whose words are all contained in a longer name (2+ words) scores 0.95."""
    ta, tb = tokens(a), tokens(b)
    if not ta or not tb:
        return 0.0
    small, big = (ta, tb) if len(ta) <= len(tb) else (tb, ta)
    if len(small) >= 2 and small <= big:
        return 0.95
    return difflib.SequenceMatcher(None, a, b).ratio()


In [ ]:
# ============================================================
# SECTION 3: Reading the files
# Every file is read by column position:
#   0 timestamp | 1 email | 2 name | 3 registration number | 4 phone | 5 attendance (not in the reference file)
# Every row becomes one dictionary with cleaned values.
# ============================================================

def read_file(path, label, has_attendance=True):
    xl = pd.ExcelFile(path)
    # use the sheet with the most rows (some files have an empty extra sheet)
    sheet = max(xl.sheet_names, key=lambda s: len(xl.parse(s, header=None)))
    df = xl.parse(sheet, dtype=object)
    headers = [str(c).split(chr(10))[0][:22] for c in df.columns[:6]]
    print(f"{label}: {len(df)} rows | columns used: {headers}")

    rows = []
    for i, r in df.iterrows():
        vals = list(r.values) + [None] * 6
        ts, email, name, reg, phone, att = vals[:6]
        if is_blank(name) and is_blank(email) and is_blank(reg) and is_blank(phone):
            continue                                   # completely empty row
        uni, nid = split_reg(reg)
        ts = pd.Timestamp(ts) if hasattr(ts, "year") else None

        # Row type:
        #  reference       = row from the reference file (no attendance info)
        #  attendance-only = no timestamp and no email (typed in manually when taking attendance)
        #  registration    = a normal form submission
        if not has_attendance:
            kind = "reference"
        elif ts is None and not norm_email(email):
            kind = "attendance-only"
        else:
            kind = "registration"

        rows.append(dict(
            source=label, row=i + 2, kind=kind,
            name=str(name).strip() if not is_blank(name) else "",
            email=norm_email(email), phone=norm_phone(phone), uni=uni, nid=nid,
            att=to_bool(att) if has_attendance else None,
            ts=ts,
        ))
    return rows


In [ ]:
# ============================================================
# SECTION 4: Deciding who is who
# Rows are linked into one person when they share a strong key.
# Priority: university ID, national ID, email, phone.
# Two rows are NEVER linked if their university IDs (or national IDs) differ.
# Names are used only for rows that have no key at all, and only when the match is unambiguous.
# ============================================================

class DSU:
    """Disjoint-set (union-find) used to group rows into people."""
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

def link_people(rows):
    n = len(rows)
    dsu = DSU(n)
    ids = {i: {"uni": {r["uni"]} - {""}, "nid": {r["nid"]} - {""}} for i, r in enumerate(rows)}
    id_conflicts = []

    def describe(i):
        r = rows[i]
        return f'{r["source"]} row {r["row"]}: {r["name"]}'

    def try_union(a, b, why):
        ra, rb = dsu.find(a), dsu.find(b)
        if ra == rb:
            return
        for k, label in (("uni", "University ID"), ("nid", "National ID")):
            if ids[ra][k] and ids[rb][k] and ids[ra][k] != ids[rb][k]:
                id_conflicts.append({
                    "Reason": why, "Row A": describe(a), "Row B": describe(b),
                    "Detail": f"{label}: {sorted(ids[ra][k])} vs {sorted(ids[rb][k])}",
                })
                return                                  # do not merge
        dsu.p[rb] = ra
        for k in ("uni", "nid"):
            ids[ra][k] |= ids[rb][k]

    # Link rows that share a key, strongest key first.
    for key, label in (("uni", "same university ID"), ("nid", "same national ID"),
                       ("email", "same email"), ("phone", "same phone")):
        seen = {}
        for i, r in enumerate(rows):
            v = r[key]
            if not v:
                continue
            if v in seen:
                try_union(seen[v], i, f"{label}: {v}")
            else:
                seen[v] = i

    # Rows with no key at all (name only): try to attach them by name.
    def names_by_group():
        g = {}
        for i, r in enumerate(rows):
            if r["name"]:
                g.setdefault(dsu.find(i), set()).add(norm_name(r["name"]))
        return g

    group_names = names_by_group()
    keyless = [i for i, r in enumerate(rows) if not (r["uni"] or r["nid"] or r["email"] or r["phone"])]
    name_links, name_review = [], []

    for i in keyless:
        nm = norm_name(rows[i]["name"])
        me = dsu.find(i)
        cands = {}
        for root, names in group_names.items():
            if root == me:
                continue
            s = max((name_similar(nm, x) for x in names), default=0)
            if s >= 0.6:
                cands[root] = s
        ranked = sorted(cands.items(), key=lambda x: -x[1])
        strong = [c for c in ranked if c[1] >= NAME_LINK_THRESHOLD]
        if len(strong) == 1 and len(tokens(nm)) >= 2:    # exactly one clear candidate, 2+ words
            target = strong[0][0]
            dsu.p[me] = target
            name_links.append({
                "Source": rows[i]["source"], "Row": rows[i]["row"], "Name": rows[i]["name"],
                "Linked to": sorted(group_names[target], key=len)[-1], "Similarity": round(strong[0][1], 2),
            })
        else:
            sug = "; ".join(f'{sorted(group_names[r], key=len)[-1]} ({s:.2f})' for r, s in ranked[:3])
            name_review.append({
                "Source": rows[i]["source"], "Row": rows[i]["row"], "Name": rows[i]["name"],
                "Attendance": rows[i]["att"], "Suggestions": sug or "none",
            })

    group_of = [dsu.find(i) for i in range(n)]
    return group_of, pd.DataFrame(id_conflicts), pd.DataFrame(name_links), pd.DataFrame(name_review)


In [ ]:
# ============================================================
# SECTION 5: Building one record per person and deciding attendance
# For every person and session we look at ALL rows of that person in that session:
#   all True   -> True
#   all False  -> False
#   mixed      -> decided by MIXED_POLICY (default: CHECK) and listed in 'Attendance conflicts'
#   no rows    -> empty (never registered for that session)
# Manual OVERRIDES always win.
# ============================================================

def build_people(rows, group_of):
    groups = {}
    for i, g in enumerate(group_of):
        groups.setdefault(g, []).append(i)

    people, evidence, conflicts, ref_only = [], [], [], []
    has_reference = any(r["kind"] == "reference" for r in rows)

    for g, idx in groups.items():
        rs = [rows[i] for i in idx]
        att_rows = [r for r in rs if r["kind"] != "reference"]
        names = [r["name"] for r in rs if r["name"]]
        # Final name = the one with the most words (then the longest text)
        best = max(names, key=lambda s: (len(s.split()), len(s))) if names else ""
        uniq = lambda k: sorted({r[k] for r in rs if r[k]})
        unis, nids = uniq("uni"), uniq("nid")
        person_ids = set(unis) | set(nids)

        rec = {
            "Name": best,
            "University ID": " | ".join(unis),
            "National ID": " | ".join(nids),
            "Email": " | ".join(uniq("email")),
            "Phone": " | ".join(uniq("phone")),
            "Other name spellings": " | ".join(sorted({x for x in names if x != best})),
        }
        notes = []
        to_check = 0
        attended = 0

        for s in SESSIONS:
            sr = [r for r in att_rows if r["source"] == s]
            if not sr:
                rec[s] = None
                continue
            values = {r["att"] for r in sr}
            status_note = ""
            if values == {True}:
                val = True
            elif values == {False}:
                val = False
            else:
                # Mixed rows: apply the policy and record the case for review.
                val = {"present": True, "absent": False}.get(MIXED_POLICY, "CHECK")
                status_note = "mixed rows"
                reg_rows = [r for r in sr if r["kind"] == "registration"]
                att_only = [r for r in sr if r["kind"] == "attendance-only"]
                late = ""
                if s in SESSION_DATES:
                    d = pd.Timestamp(SESSION_DATES[s]).date()
                    lates = [r for r in reg_rows if r["ts"] is not None and r["ts"].date() > d]
                    late = "yes" if lates else "no"
                owner_dec = "Attended (confirmed manually)" if MIXED_POLICY == "present" else "Pending"
                conflicts.append({
                    "Person": best, "ID": (unis + nids + [""])[0], "Session": s,
                    "Registration rows": "; ".join(f'row {r["row"]} = {r["att"]}' for r in reg_rows) or "none",
                    "Attendance-only rows": "; ".join(f'row {r["row"]} = {r["att"]} (as "{r["name"]}")' for r in att_only) or "none",
                    "Registered after session date": late or "not checked",
                    "Resolved as": str(val),
                    "Override key": f'("{(unis + nids + [""])[0]}", "{s}")',
                    "Owner decision": owner_dec,
                })
            # Manual override
            for pid in person_ids:
                if (pid, s) in OVERRIDES:
                    val = bool(OVERRIDES[(pid, s)])
                    status_note = "manual override"
                    break

            rec[s] = val
            if val is True:
                attended += 1
            if val == "CHECK":
                to_check += 1
                notes.append(f"{s}: check")
            elif status_note == "mixed rows" and MIXED_POLICY == "present":
                notes.append(f"{s}: mixed rows, confirmed attended by owner")
            elif status_note == "manual override":
                notes.append(f"{s}: override")

            # Late-registration flag (only when SESSION_DATES is set)
            late_rows = []
            if s in SESSION_DATES:
                d = pd.Timestamp(SESSION_DATES[s]).date()
                late_rows = [r for r in sr if r["kind"] == "registration" and r["ts"] is not None and r["ts"].date() > d]
                if late_rows:
                    notes.append(f"{s}: registered after session date")

            for r in sr:
                evidence.append({
                    "Person": best, "ID": (unis + nids + [""])[0], "Session": s, "Source row": r["row"],
                    "Row type": r["kind"], "Name as written": r["name"], "Attendance as written": r["att"],
                    "Timestamp": r["ts"].strftime("%Y-%m-%d %H:%M") if r["ts"] is not None else "",
                    "Session result": str(val), "Flag": status_note,
                })

        rec["In reference list"] = (True if any(r["kind"] == "reference" for r in rs) else None) if has_reference else None
        rec["Sessions attended"] = attended
        rec["Sessions to check"] = to_check
        rec["Notes"] = " / ".join(notes)

        if len(unis) > 1 or len(nids) > 1:
            rec["Notes"] = (rec["Notes"] + " / " if rec["Notes"] else "") + "multiple IDs"

        if not att_rows:                              # person exists only in the reference file
            ref_only.append({k: rec[k] for k in ("Name", "University ID", "National ID", "Email", "Phone")})
            if not INCLUDE_REFERENCE_ONLY_PEOPLE:
                continue
        people.append(rec)

    master = pd.DataFrame(people)
    if not has_reference:
        master = master.drop(columns=["In reference list"])
    master = master.sort_values(["Sessions attended", "Name"], ascending=[False, True]).reset_index(drop=True)
    return master, pd.DataFrame(evidence), pd.DataFrame(conflicts), pd.DataFrame(ref_only)

def find_possible_duplicates(master):
    """Pairs of people with very similar multi-word names that were NOT merged (no shared key)."""
    recs = master.to_dict("records")
    out = []
    for a in range(len(recs)):
        for b in range(a + 1, len(recs)):
            na, nb = norm_name(recs[a]["Name"]), norm_name(recs[b]["Name"])
            if na and nb and len(tokens(na)) >= 2 and name_similar(na, nb) >= NAME_LINK_THRESHOLD:
                out.append({
                    "Name A": recs[a]["Name"], "ID A": recs[a]["University ID"] or recs[a]["National ID"],
                    "Name B": recs[b]["Name"], "ID B": recs[b]["University ID"] or recs[b]["National ID"],
                })
    return pd.DataFrame(out)


In [ ]:
# ============================================================
# SECTION 6: Excel export
# One workbook with the Master sheet first, then the review sheets.
# Colours in the session columns: green = True, red = False, yellow = CHECK.
# ============================================================
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

def save_excel(path, sheets):
    """sheets = ordered dict {sheet_name: DataFrame}"""
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        for name, df in sheets.items():
            (df if len(df) else pd.DataFrame({"Info": ["No cases"]})).to_excel(w, sheet_name=name, index=False)

        for ws in w.book.worksheets:
            for c in ws[1]:
                c.font = Font(name="Arial", bold=True, color="FFFFFF")
                c.fill = PatternFill("solid", fgColor="1F4E78")
                c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            ws.freeze_panes = "B2"
            ws.auto_filter.ref = ws.dimensions
            for col in ws.columns:
                width = max(len(str(c.value)) if c.value is not None else 0 for c in col)
                ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(12, width + 2), 50)
            for row in ws.iter_rows(min_row=2):
                for c in row:
                    c.font = Font(name="Arial")

        green = PatternFill("solid", fgColor="C6EFCE")
        red = PatternFill("solid", fgColor="FFC7CE")
        yellow = PatternFill("solid", fgColor="FFEB9C")
        ws = w.book["Master"]
        for row in ws.iter_rows(min_row=2):
            for c in row:
                if c.value is True:
                    c.fill = green
                elif c.value is False:
                    c.fill = red
                elif c.value == "CHECK":
                    c.fill = yellow


## 4. Run and download

In [ ]:
# ============================================================
# SECTION 7: Run everything and download the result
# ============================================================
SESSIONS = [s for s in SESSIONS if s in SESSION_FILES]
assert SESSIONS, "Upload at least one session file first."

# 1) Read all files
all_rows = []
for s in SESSIONS:
    all_rows += read_file(SESSION_FILES[s], s, has_attendance=True)
if REFERENCE_LABEL in SESSION_FILES:
    all_rows += read_file(SESSION_FILES[REFERENCE_LABEL], REFERENCE_LABEL, has_attendance=False)

# 2) Decide who is who
group_of, id_conflicts, name_links, name_review = link_people(all_rows)

# 3) Build people and attendance
master, evidence, att_conflicts, ref_only = build_people(all_rows, group_of)
duplicates = find_possible_duplicates(master)

# 4) Save to Excel
OUT = "Master_Attendance_final.xlsx"
save_excel(OUT, {
    "Master": master,
    "Attendance conflicts": att_conflicts,
    "Evidence": evidence,
    "ID conflicts": id_conflicts,
    "Name review": name_review,
    "Name links": name_links,
    "Possible duplicates": duplicates,
    "Reference only": ref_only,
})

# 5) Summary
print(f"\nPeople in Master: {len(master)} (from {len(all_rows)} rows)")
print(f"Attendance cells to CHECK: {len(att_conflicts)}")
print(f"ID conflicts: {len(id_conflicts)} | Name-only rows to review: {len(name_review)} | Possible duplicates: {len(duplicates)}")
if len(ref_only):
    print(f"People only in the reference file: {len(ref_only)} (sheet 'Reference only')")
display(master.head(20))
files.download(OUT)
